# ResNet1D on raw EMG waveforms: training

Trains Wang et al. 2017's ResNet (`resnet1d.py`) on per-trial z-scored raw EMG (`loadData.py`) for Subject 1.
Settings and their sources are in `../../PLAN.md` (Step 3). Run from inside `training_runs/subject_01/`.

**How to use:** run the Setup, Training functions and Graphing sections once. Then each of sections 1–4 trains one
model type and graphs it, and can be run on its own. Everything a run produces goes to `results/`:
`results.json` (rewritten after each section, so
sections from earlier sessions are kept; **Overall statistics** at the end summarizes everything saved so far),
`progress.log` (appended to; `tail -f results/progress.log` to follow it), and one trained model per run.

## Setup

In [ ]:
import json, os, sys, time, warnings

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

sys.path.insert(0, "../..")  # resnet1d/, for loadData.py and resnet1d.py
from loadData import loadZscoredTrainTestSplit
from resnet1d import resnet1dNet

# PyTorch warns that padding = "same" with the even kernel of 8 makes an internal padded copy; results are unaffected.
warnings.filterwarnings("ignore", message = "Using padding='same' with even kernel lengths")

In [ ]:
numberSeeds = 5
numberEpochs = 300
batchSize = 32
device = torch.device("cuda:0")

print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
def report(line):
    print(line)
    with open("results/progress.log", "a") as logFile:
        logFile.write(line + "\n")


# Results saved by earlier sessions are loaded, so any section can be (re)run on its own without losing the others.
results = json.load(open("results/results.json")) if os.path.exists("results/results.json") else {}
print("configs already in results.json:", list(results) or "none")

## Training functions

- `loadTrialsOntoGpu`: loads one recording with `loadData.py` (per-trial z-scoring, fixed 6/4 split) and moves it to the GPU.
- `evaluate`: test loss and accuracy, in eval mode, without gradients.
- `trainOneRun`: trains one fresh model for `numberEpochs` and returns its per-epoch history, plus a copy of the weights
  from the epoch with the highest test accuracy.
- `trainAllSeeds`: runs `trainOneRun` for every seed, reads the three test accuracies off each history, saves each run's
  model to `results/<config>_seed<N>_bestTestAccuracy.pt`, and saves everything else to `results/results.json`.

In [ ]:
def loadTrialsOntoGpu(taskFolder, articulationManner):
    """taskFolder is "Phoneme" or "Words"; articulationManner is "Voiced" or "Unvoiced"."""
    trainTrials, trainLabels, testTrials, testLabels = loadZscoredTrainTestSplit(taskFolder, articulationManner)
    print(f"{taskFolder} {articulationManner}: train {trainTrials.shape}, test {testTrials.shape}")
    return [torch.from_numpy(array).to(device) for array in (trainTrials, trainLabels, testTrials, testLabels)]

In [ ]:
def evaluate(model, trials, labels, lossFunction):
    """Mean loss and accuracy in eval mode (BatchNorm uses its running statistics, so trials don't affect each other)."""
    model.eval()
    lossSum, correctCount = 0.0, 0

    with torch.no_grad():
        for start in range(0, len(labels), batchSize):
            scores = model(trials[start:start + batchSize])
            batchLabels = labels[start:start + batchSize]

            lossSum += lossFunction(scores, batchLabels).item() * len(batchLabels)
            correctCount += (scores.argmax(dim = 1) == batchLabels).sum().item()

    return lossSum / len(labels), correctCount / len(labels)

In [ ]:
def trainOneRun(trainTrials, trainLabels, testTrials, testLabels, seed):
    torch.manual_seed(seed)
    model = resnet1dNet(numberClasses = int(trainLabels.max()) + 1).to(device)
    lossFunction = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.001, betas = (0.9, 0.999), eps = 1e-8)
    # From Wang et al.'s code: halve the learning rate when training loss hasn't improved for 50 epochs, down to 1e-4.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = "min", factor = 0.5, patience = 50, min_lr = 1e-4)

    history = {"trainLoss": [], "trainAccuracy": [], "testLoss": [], "testAccuracy": [], "learningRate": []}
    # Weights from the epoch with the highest test accuracy (first one on ties, like np.argmax), copied to the CPU.
    bestTestAccuracy, bestTestAccuracyWeights = -1, None

    report(" epoch  train loss  train acc  test loss  test acc       lr  s/epoch")
    startTime = time.time()
    for epoch in range(numberEpochs):
        model.train() # Set to train mode
        shuffledOrder = torch.randperm(len(trainLabels), device = device)
        lossSum, correctCount = 0.0, 0

        for start in range(0, len(shuffledOrder), batchSize):
            batchIndices = shuffledOrder[start:start + batchSize]
            scores = model(trainTrials[batchIndices]) # Gets inference scores in the suffled order
            loss = lossFunction(scores, trainLabels[batchIndices])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            lossSum += loss.item() * len(batchIndices)
            correctCount += (scores.argmax(dim = 1) == trainLabels[batchIndices]).sum().item()

        # Training loss = average over this epoch's batches in train mode, like the Keras "loss" Wang's rule selects on.
        epochMetrics = {"trainLoss": lossSum / len(trainLabels),
                        "trainAccuracy": correctCount / len(trainLabels),
                        "learningRate": optimizer.param_groups[0]["lr"]}
        scheduler.step(epochMetrics["trainLoss"])
        epochMetrics["testLoss"], epochMetrics["testAccuracy"] = evaluate(model, testTrials, testLabels, lossFunction)

        for key, value in epochMetrics.items():
            history[key].append(value)

        if epochMetrics["testAccuracy"] > bestTestAccuracy:
            bestTestAccuracy = epochMetrics["testAccuracy"]
            bestTestAccuracyWeights = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}

        # Print out status every 10 epochs
        if (epoch + 1) % 10 == 0:
            m = epochMetrics
            report(f"{epoch + 1:>6}  {m['trainLoss']:>10.4f}  {m['trainAccuracy']:>9.3f}  {m['testLoss']:>9.4f}  {m['testAccuracy']:>8.3f}"
                   f"  {m['learningRate']:>7.1e}  {(time.time() - startTime) / (epoch + 1):>7.2f}")

    return history, bestTestAccuracyWeights

In [ ]:
# The three ways of turning a run's per-epoch test accuracies into one number (see PLAN.md, Step 3).
accuracyRuleLabels = {"testAccuracyAtLowestTrainLoss": "at lowest train loss (Wang)",
                      "bestTestAccuracy": "best across epochs (repo)",
                      "finalTestAccuracy": "final epoch"}


def trainAllSeeds(configName, trainTrials, trainLabels, testTrials, testLabels):
    runs = []
    for seed in range(numberSeeds):
        report(f"=== {configName}, seed {seed}")

        history, bestTestAccuracyWeights = trainOneRun(trainTrials, trainLabels, testTrials, testLabels, seed)

        modelPath = f"results/{configName}_seed{seed}_bestTestAccuracy.pt"
        torch.save(bestTestAccuracyWeights, modelPath)

        lowestTrainLossEpoch = int(np.argmin(history["trainLoss"]))
        bestTestEpoch = int(np.argmax(history["testAccuracy"]))
        run = {"seed": seed,
               "testAccuracyAtLowestTrainLoss": history["testAccuracy"][lowestTrainLossEpoch],
               "lowestTrainLossEpoch": lowestTrainLossEpoch + 1,
               "bestTestAccuracy": history["testAccuracy"][bestTestEpoch],
               "bestTestAccuracyEpoch": bestTestEpoch + 1,
               "finalTestAccuracy": history["testAccuracy"][-1],
               "modelPath": modelPath,
               "history": history}

        report(f"test accuracy, {configName} seed {seed}:")
        report(f"  {accuracyRuleLabels['testAccuracyAtLowestTrainLoss']:<28} {run['testAccuracyAtLowestTrainLoss']:.3f}  (epoch {run['lowestTrainLossEpoch']})")
        report(f"  {accuracyRuleLabels['bestTestAccuracy']:<28} {run['bestTestAccuracy']:.3f}  (epoch {run['bestTestAccuracyEpoch']})")
        report(f"  {accuracyRuleLabels['finalTestAccuracy']:<28} {run['finalTestAccuracy']:.3f}")
        report(f"  saved model from epoch {run['bestTestAccuracyEpoch']} to {modelPath}")
        runs.append(run)

    results[configName] = {"numberClasses": int(trainLabels.max()) + 1,
                           "numberTestTrials": len(testLabels),
                           "numberEpochs": numberEpochs,
                           "batchSize": batchSize,
                           "numberSeeds": numberSeeds,
                           "runs": runs}
    with open("results/results.json", "w") as resultsFile:
        json.dump(results, resultsFile)

## Graphing function

`plotTrainingCurves` draws one config's train/test accuracy and training loss over epochs
(mean over seeds, shaded min–max when there is more than one seed). It reads from `results`, so it also works for configs saved to `results/results.json` in an earlier session.

In [ ]:
def plotTrainingCurves(configName):
    config = results[configName]
    epochNumbers = np.arange(1, config["numberEpochs"] + 1)
    fig, (accuracyAxis, lossAxis) = plt.subplots(1, 2, figsize = (12, 3.5))

    # Left: train and test accuracy
    for key, label in [("trainAccuracy", "train"), ("testAccuracy", "test")]:
        valuesBySeed = np.array([run["history"][key] for run in config["runs"]])
        accuracyAxis.plot(epochNumbers, valuesBySeed.mean(0), label = label)
        accuracyAxis.fill_between(epochNumbers, valuesBySeed.min(0), valuesBySeed.max(0), alpha = 0.2)
    accuracyAxis.axhline(1 / config["numberClasses"], color = "gray", ls = ":", lw = 1, label = "chance")
    accuracyAxis.set(title = configName + ": accuracy", xlabel = "epoch", ylabel = "accuracy")
    accuracyAxis.legend(fontsize = 8)

    # Right: training loss, log scale
    trainLossBySeed = np.array([run["history"]["trainLoss"] for run in config["runs"]])
    lossAxis.semilogy(epochNumbers, trainLossBySeed.mean(0))
    lossAxis.set(title = configName + ": training loss", xlabel = "epoch", ylabel = "train loss (log)")

    plt.tight_layout()
    plt.show()

## 1. All phonemes, voiced

In [ ]:
trainTrials, trainLabels, testTrials, testLabels = loadTrialsOntoGpu("Phoneme", "Voiced")
trainAllSeeds("allPhonemes_Voiced", trainTrials, trainLabels, testTrials, testLabels)

In [ ]:
plotTrainingCurves("allPhonemes_Voiced")

## 2. All phonemes, unvoiced

In [ ]:
trainTrials, trainLabels, testTrials, testLabels = loadTrialsOntoGpu("Phoneme", "Unvoiced")
trainAllSeeds("allPhonemes_Unvoiced", trainTrials, trainLabels, testTrials, testLabels)

In [ ]:
plotTrainingCurves("allPhonemes_Unvoiced")

## 3. All words, voiced

In [ ]:
trainTrials, trainLabels, testTrials, testLabels = loadTrialsOntoGpu("Words", "Voiced")
trainAllSeeds("allWords_Voiced", trainTrials, trainLabels, testTrials, testLabels)

In [ ]:
plotTrainingCurves("allWords_Voiced")

## 4. All words, unvoiced

In [ ]:
trainTrials, trainLabels, testTrials, testLabels = loadTrialsOntoGpu("Words", "Unvoiced")
trainAllSeeds("allWords_Unvoiced", trainTrials, trainLabels, testTrials, testLabels)

In [ ]:
plotTrainingCurves("allWords_Unvoiced")

## Overall statistics

Test accuracy per config under each rule: mean ± std over seeds (std is 0 with one seed). Uses everything in `results/results.json`.

In [ ]:
print(f"{'config':<22} {'classes':>7} {'chance':>7} {'seeds':>5} {'epochs':>6}   "
      + "   ".join(f"{label:>27}" for label in accuracyRuleLabels.values()))

for configName, config in results.items():
    accuracyCells = []
    for ruleKey in accuracyRuleLabels:
        accuracies = np.array([run[ruleKey] for run in config["runs"]])
        accuracyCells.append(f"{accuracies.mean():.3f} ± {accuracies.std():.3f}")

    print(f"{configName:<22} {config['numberClasses']:>7} {1 / config['numberClasses']:>7.3f} {config['numberSeeds']:>5} {config['numberEpochs']:>6}   "
          + "   ".join(f"{accuracyCell:>27}" for accuracyCell in accuracyCells))